### Bibliotecas

In [1]:
import numpy as np
import pandas as pd 

from sklearn.model_selection import KFold # Pra validación cruzada
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

from sklearn.metrics import precision_score, confusion_matrix

# Para usar datos de CSVs, Ordinal es para datos y Label para clases
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder

### Lee datos

In [5]:
df = pd.read_csv('agaricus-lepiota.csv')
df.head(7)

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g
5,e,x,y,y,t,a,f,c,b,n,...,s,w,w,p,w,o,p,k,n,g
6,e,b,s,w,t,a,f,c,b,g,...,s,w,w,p,w,o,p,k,n,m


### Haz X(datos) y y (Clases)

In [8]:
Xn = df.drop(columns=['class'], axis=1)
yn =df['class']

print(Xn.head(2), yn.head(2))

  cap-shape cap-surface cap-color bruises odor gill-attachment gill-spacing  \
0         x           s         n       t    p               f            c   
1         x           s         y       t    a               f            c   

  gill-size gill-color stalk-shape  ... stalk-surface-below-ring  \
0         n          k           e  ...                        s   
1         b          k           e  ...                        s   

  stalk-color-above-ring stalk-color-below-ring veil-type veil-color  \
0                      w                      w         p          w   
1                      w                      w         p          w   

  ring-number ring-type spore-print-color population habitat  
0           o         p                 k          s       u  
1           o         p                 n          n       g  

[2 rows x 22 columns] 0    p
1    e
Name: class, dtype: object


### Codifica con Numero

In [9]:
encX = OrdinalEncoder() # Para datos
encX.fit(Xn)
X = encX.transform(Xn)

ency = LabelEncoder() # Para clases
ency.fit(yn)
y = ency.transform(yn)

print(X[0:2], y[0:2])

[[5. 2. 4. 1. 6. 1. 0. 1. 4. 0. 3. 2. 2. 7. 7. 0. 2. 1. 4. 2. 3. 5.]
 [5. 2. 9. 1. 0. 1. 0. 0. 4. 0. 2. 2. 2. 7. 7. 0. 2. 1. 4. 3. 2. 1.]] [1 0]


### Aprende con todos los datos

In [14]:
tree = DecisionTreeClassifier(criterion='entropy') # Modelo
tree.fit(X, y) # Aprende

svm = SVC(kernel='poly',degree=2) # Modelo
svm.fit(X, y) # Aprende

,C,1.0
,kernel,'poly'
,degree,2
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


### Reporta resultados de aprender con todos

In [15]:
precision_tree = precision_score(y, tree.predict(X))
precision_svm = precision_score(y, svm.predict(X))

print(f'Precisión del Árbol de Decisión: {precision_tree}')
print(f'Con matriz de confusión:\n{confusion_matrix(y, tree.predict(X))}')
print(f'\n\nPrecisión del SVM: {precision_svm}')
print(f'Con matriz de confusión:\n{confusion_matrix(y, svm.predict(X))}')

Precisión del Árbol de Decisión: 1.0
Con matriz de confusión:
[[4208    0]
 [   0 3916]]


Precisión del SVM: 0.9940321743642968
Con matriz de confusión:
[[4185   23]
 [  85 3831]]


## Validación cruzada de diez (K) mangas _KFold_

### KFold con _Árboles de Decisión_

In [17]:
n = 10
kf = KFold(n_splits=n, shuffle=True, random_state=7)
promedio = 0.0
for i, (train, test) in enumerate(kf.split(X)):
    tree.fit(X[train], y[train]) # Aprende con los datos elegidos
    precision = precision_score(y[test], tree.predict(X[test])) # Prueba para ver que tan bien aprendió
    promedio += precision
    print(f'La precisión en la manga {i+1} es {precision}')

promedio /= n
print(f'La precisión promedio en las {n} mangas es {promedio}')


La precisión en la manga 1 es 1.0
La precisión en la manga 2 es 1.0
La precisión en la manga 3 es 1.0
La precisión en la manga 4 es 1.0
La precisión en la manga 5 es 1.0
La precisión en la manga 6 es 1.0
La precisión en la manga 7 es 1.0
La precisión en la manga 8 es 1.0
La precisión en la manga 9 es 1.0
La precisión en la manga 10 es 1.0
La precisión promedio en las 10 mangas es 1.0


## KFold con _SVM_

In [20]:
n = 10
kf = KFold(n_splits=n, shuffle=True, random_state=7)
promedio = 0.0
for i, (train, test) in enumerate(kf.split(X)):
    svm.fit(X[train], y[train]) # Aprende con los datos elegidos
    precision = precision_score(y[test], svm.predict(X[test])) # Prueba para ver que tan bien aprendió
    promedio += precision
    print(f'La precisión en la manga {i+1} es {precision:.4f}')

promedio /= n
print(f'La precisión promedio en las {n} mangas es {promedio:.4f}')

La precisión en la manga 1 es 0.9948
La precisión en la manga 2 es 0.9974
La precisión en la manga 3 es 0.9825
La precisión en la manga 4 es 0.9950
La precisión en la manga 5 es 0.9948
La precisión en la manga 6 es 0.9865
La precisión en la manga 7 es 0.9771
La precisión en la manga 8 es 0.9974
La precisión en la manga 9 es 0.9944
La precisión en la manga 10 es 0.9817
La precisión promedio en las 10 mangas es 0.9902
